# CLLM generation & curation tutorial

# Setup keys and experiment

In [1]:
import sys
sys.path.append('src/')

from cllm.utils import *
from cllm.llm_gen import *
from cllm.data_loader import *
import numpy as np

import pandas as pd
from copy import deepcopy
import time
from sklearn.model_selection import train_test_split


#############################################################
# API KEY SETUP INSTRUCTIONS
#############################################################

# for vllm
# api_key = "EMPTY"
# api_base = "http://localhost:8000/v1"

# for together
# api_key = "add together api key"
# api_base = "https://api.together.xyz/v1"


# for azure openai
# api_key = "EMPTY"
# api_base = "add azure deployment link"

# for openai
# api_key = "EMPTY"
# api_base = DO NOT INCLUDE

#############################################################

api_details = {
     "api_base": "https://api.together.xyz/v1",
     "api_version": "2023-07-01-preview",
     "api_key": "91a65a9f1f3d858e4d7f4de4fdf14fde879712b254c81ca07c0cacc707b59d82",
}


model_short_name = 'mixtral' # 'gpt-4' (do not use other short names)
model = "mistralai/Mixtral-8x7B-Instruct-v0.1" # "gpt4_20230815" (use name of your model deployment)
llm_serving='together' # supported 'azure_openai', 'together', 'vllm'

seed = 0
ns = 20 # n_samples per class. e.g. if binary = 40 samples (i.e. 20 per class)
dataset = 'compas'
n_synthetic=100 # just to test --- normall should be 1000
n_processes = 5

# STEP 1: Generation

## Get dataset

In [ ]:
data = pd.read_csv("mtbls547/mtbls547.csv")
data.rename(columns={'Class': 'y'}, inplace=True)
data = data.drop(['Idx','SampleID','SampleType', 'Chronic Fatigue Syndrome'], axis=1)
data = data.applymap(lambda x: float(x.replace(',', '')) if isinstance(x, str) and x.replace(',', '').replace('.', '').isdigit() else x)
data['y'] = data['y'].astype(int)
df_feat = data.drop('y', axis=1)
df_label = data["y"]
df = data
count_0 = (df['y'] == 0).sum()
count_1 = (df['y'] == 1).sum()

print(f"Count of y=0: {count_0}")
print(f"Count of y=1: {count_1}")
needed_synthetic = count_1 - count_0

Count of y=0: 25
Count of y=1: 34


In [61]:
#df_feat, df_label, df = get_data(dataset=dataset, seed=seed)
df_feat.to_csv("dffeat")
df_label.to_csv("df_label")
df.to_csv("df")

X_train, X_remain, y_train, y_remain = sample_and_split(df_feat, df_label, ns=ns, seed=seed)

X_val, X_test, y_val, y_test = train_test_split(
    X_remain, y_remain, test_size=0.5, random_state=seed
)


X_train_orig = deepcopy(X_train)
y_train_orig = deepcopy(y_train)

## Setup Prompt

In [62]:
from langchain.prompts import ChatPromptTemplate
from langchain.output_parsers import ResponseSchema
from langchain.output_parsers import StructuredOutputParser


response_schemas = []

example_df = pd.concat([X_train_orig, y_train_orig], axis=1)

# Shuffle
example_df = example_df.sample(frac=1).reset_index(drop=True)


for idx, col in enumerate(list(example_df.columns)):
    if col == 'y':
        resp = ResponseSchema(name='y',
                        description=f"binary label, {col}", )
    else:
        resp = ResponseSchema(name=col,
                        description=f"feature column", )
    response_schemas.append(resp)

output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()



generator_template = """\
You are a synthetic data generator. 
Your goal is to produce data which mirrors \
the given examples in causal structure and feature and label distributions \
but also produces as diverse samples as possible

I will give you real examples first

Leverage your knowledge about chronic fatique syndrome to generate 1000 realistic but diverse samples. 

example data: {data}

{format_instructions}

DO NOT COPY THE EXAMPLES but generate realistic but new and diverse samples which have the correct label conditioned on the features.
"""


prompt = ChatPromptTemplate.from_template(template=generator_template)

## Generate using LLM

In [63]:
retries = 4  # Max retries you want to attempt
df_llm = pd.DataFrame(columns=example_df.columns)
iters = 0
while True:
    print(f"Iter {iters}")
    print(f"Currently {df_llm.shape[0]} samples")
    while retries > 0:
        try:

            if len(example_df)>20:
                ic_samples=20
            else:
                ic_samples=len(example_df)
            
            #print(f'Running {dataset}, {seed}, {model} --- {n_processes}')
            tmp_llm = llm_gen(prompt, generator_template, format_instructions, example_df, 
                            n_samples=needed_synthetic*2,
                            temperature=0.9,
                            max_tokens=1000, model=model, 
                            n_processes=n_processes,
                            ic_samples=ic_samples, 
                            llm_serving=llm_serving, 
                            api_details=api_details)
        
            
            break  # if successful, break out of the loop
        except Exception as e:
            time.sleep(120)
            print(f"Error: {e}. Retrying with reduced n_processes...")
            n_processes = int(n_processes/2)
            retries -= 1
            if n_processes < 1:
                print("Error: Minimum n_processes reached. Exiting...")
                break
    # try:
    tmp_df = tmp_llm.astype(example_df.dtypes)
    tmp_df = tmp_df[tmp_df['y'] == 0]
    df_llm = pd.concat([df_llm, tmp_df], ignore_index=True)
    if df_llm.shape[0] >= needed_synthetic:
        break
# except:
#     pass

Iter 0
Currently 0 samples
idx: 0
df_tmp Empty DataFrame
Columns: []
Index: []
idx: 1
idx: 2
idx: 3
idx: 4
Current =  3 (3, 30)


c:\Users\chitt\OneDrive\Documents\Desktop\2200\docker\docker\workspace\ta\CLLM\src\cllm\llm_gen.py:255: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_tmp = df_tmp.append(df_check, ignore_index=True)
c:\Users\chitt\OneDrive\Documents\Desktop\2200\docker\docker\workspace\ta\CLLM\src\cllm\llm_gen.py:255: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_tmp = df_tmp.append(df_check, ignore_index=True)
c:\Users\chitt\OneDrive\Documents\Desktop\2200\docker\docker\workspace\ta\CLLM\src\cllm\llm_gen.py:255: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_tmp = df_tmp.append(df_check, ignore_index=True)
c:\Users\chitt\OneDrive\Documents\Desktop\2200\docker\docker\workspace\ta\CLLM\src\cllm\llm_gen.py:255: FutureWarning: The fram

idx: 0
df_tmp Empty DataFrame
Columns: []
Index: []
idx: 1
idx: 2
idx: 3
idx: 4
Current =  9 (9, 30)


c:\Users\chitt\OneDrive\Documents\Desktop\2200\docker\docker\workspace\ta\CLLM\src\cllm\llm_gen.py:255: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_tmp = df_tmp.append(df_check, ignore_index=True)
c:\Users\chitt\OneDrive\Documents\Desktop\2200\docker\docker\workspace\ta\CLLM\src\cllm\llm_gen.py:255: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_tmp = df_tmp.append(df_check, ignore_index=True)
c:\Users\chitt\OneDrive\Documents\Desktop\2200\docker\docker\workspace\ta\CLLM\src\cllm\llm_gen.py:255: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_tmp = df_tmp.append(df_check, ignore_index=True)
c:\Users\chitt\OneDrive\Documents\Desktop\2200\docker\docker\workspace\ta\CLLM\src\cllm\llm_gen.py:255: FutureWarning: The fram

idx: 0
df_tmp Empty DataFrame
Columns: []
Index: []
idx: 1
idx: 2
idx: 3
idx: 4
Current =  12 (12, 30)


...

c:\Users\chitt\OneDrive\Documents\Desktop\2200\docker\docker\workspace\ta\CLLM\src\cllm\llm_gen.py:255: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_tmp = df_tmp.append(df_check, ignore_index=True)
c:\Users\chitt\OneDrive\Documents\Desktop\2200\docker\docker\workspace\ta\CLLM\src\cllm\llm_gen.py:255: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_tmp = df_tmp.append(df_check, ignore_index=True)
c:\Users\chitt\OneDrive\Documents\Desktop\2200\docker\docker\workspace\ta\CLLM\src\cllm\llm_gen.py:255: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_tmp = df_tmp.append(df_check, ignore_index=True)
c:\Users\chitt\OneDrive\Documents\Desktop\2200\docker\docker\workspace\ta\CLLM\src\cllm\llm_gen.py:255: FutureWarning: The fram

idx: 0
df_tmp Empty DataFrame
Columns: []
Index: []
idx: 1
idx: 2
idx: 3
idx: 4
Current =  13 (13, 30)


c:\Users\chitt\OneDrive\Documents\Desktop\2200\docker\docker\workspace\ta\CLLM\src\cllm\llm_gen.py:255: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_tmp = df_tmp.append(df_check, ignore_index=True)
c:\Users\chitt\OneDrive\Documents\Desktop\2200\docker\docker\workspace\ta\CLLM\src\cllm\llm_gen.py:255: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_tmp = df_tmp.append(df_check, ignore_index=True)
c:\Users\chitt\OneDrive\Documents\Desktop\2200\docker\docker\workspace\ta\CLLM\src\cllm\llm_gen.py:255: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_tmp = df_tmp.append(df_check, ignore_index=True)
c:\Users\chitt\OneDrive\Documents\Desktop\2200\docker\docker\workspace\ta\CLLM\src\cllm\llm_gen.py:255: FutureWarning: The fram

idx: 0
df_tmp Empty DataFrame
Columns: []
Index: []
idx: 1
idx: 2
idx: 3
idx: 4
Current =  21 (21, 30)
Done...
21 (21, 30)


c:\Users\chitt\OneDrive\Documents\Desktop\2200\docker\docker\workspace\ta\CLLM\src\cllm\llm_gen.py:255: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_tmp = df_tmp.append(df_check, ignore_index=True)
c:\Users\chitt\OneDrive\Documents\Desktop\2200\docker\docker\workspace\ta\CLLM\src\cllm\llm_gen.py:255: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_tmp = df_tmp.append(df_check, ignore_index=True)
c:\Users\chitt\OneDrive\Documents\Desktop\2200\docker\docker\workspace\ta\CLLM\src\cllm\llm_gen.py:255: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_tmp = df_tmp.append(df_check, ignore_index=True)
c:\Users\chitt\OneDrive\Documents\Desktop\2200\docker\docker\workspace\ta\CLLM\src\cllm\llm_gen.py:255: FutureWarning: The fram

In [64]:
df_llm


,M1,M2,M3,M4,M5,M6,M7,M8,M9,M10,...,M21,M22,M23,M24,M25,M26,M27,M28,M29,y
0,8.500000,136.200000,40.100000,24.600000,15.500000,11.900000,20.100000,21.200000,5.600000,18.800000,...,56.800000,30.200000,16.500000,63.600000,42.600000,46.100000,17.800000,18.900000,68.90000,0
1,8.500000,155.200000,48.100000,22.600000,18.200000,12.600000,25.200000,18.300000,5.600000,16.500000,...,55.300000,41.900000,21.300000,64.200000,51.800000,51.400000,17.300000,16.800000,61.60000,0
2,8.900000,136.200000,47.600000,23.900000,18.200000,12.600000,24.100000,41.300000,11.500000,12.600000,...,58.200000,32.400000,17.100000,61.200000,37.900000,41.200000,19.700000,20.900000,51.80000,0
3,8.500000,126.100000,44.600000,23.900000,18.200000,13.900000,21.500000,23.700000,8.100000,21.200000,...,59.800000,37.200000,17.300000,64.500000,38.200000,48.600000,15.700000,17.900000,54.10000,0
4,8.900000,145.200000,44.500000,18.200000,15.600000,9.400000,18.300000,16.200000,6.800000,17.300000,...,55.100000,32.100000,15.200000,67.800000,38.900000,30.600000,11.200000,15.900000,57.30000,0
5,8.100000,118.200000,38.600000,28.300000,15.600000,17.200000,18.900000,23.300000,12.600000,20.500000,...,56.100000,37.300000,19.100000,61.100000,48.200000,47.500000,16.200000,19.300000,56.90000,0
6,9.300000,115.200000,37.500000,18.900000,15.600000,10.800000,23.700000,17.300000,12.100000,20.600000,...,50.700000,32.900000,14.500000,61.600000,48.200000,47.900000,16.200000,18.800000,60.50000,0
7,13.400000,158.900000,43.200000,26.900000,16.100000,7.200000,23.700000,13.500000,7.200000,16.800000,...,46.800000,35.400000,21.400000,43.800000,44.900000,35.200000,14.300000,15.600000,51.50000,0
8,6.400000,112.800000,45.600000,24.300000,15.900000,11.200000,18.900000,18.800000,4.100000,16.500000,...,51.600000,34.400000,15.200000,54.900000,32.700000,48.100000,14.800000,18.200000,53.50000,0
9,13.200000,145.900000,57.500000,30.900000,20.600000,13.700000,25.300000,27.600000,7.800000,14.400000,...,54.300000,39.800000,19.600000,37.200000,26.400000,39.500000,12.900000,16.300000,45.20000,0


In [65]:
len(df_llm)
print(df_llm.dtypes)

M1     float64
M2     float64
M3     float64
M4     float64
M5     float64
M6     float64
M7     float64
M8     float64
M9     float64
M10    float64
M11    float64
M12    float64
M13    float64
M14    float64
M15    float64
M16    float64
M17    float64
M18    float64
M19    float64
M20    float64
M21    float64
M22    float64
M23    float64
M24    float64
M25    float64
M26    float64
M27    float64
M28    float64
M29    float64
y       object
dtype: object


## Process LLM generated data to have the same data types

In [66]:
df_llm = df_llm.dropna()
df_llm = df_llm[~df_llm.apply(lambda row: any([isinstance(cell, str) and cell in ['integer', 'float', 'numeric', 'categorical', 'number', 'No', 'Yes', 'continuous', 'age in years', 'string'] for cell in row]), axis=1)]

example_df = deepcopy(X_train_orig)
example_df['y'] = deepcopy(y_train_orig)

try:
    df_llm = df_llm.astype(example_df.dtypes)
except:
    # Assuming the dtypes from the example_df['Dtrain'].dataframe() is what you want
    target_dtypes = example_df.dtypes.to_dict()

    problematic_rows = set()

    for col, dtype in target_dtypes.items():
        for index, value in df[col].items():
            try:
                _ = dtype.type(value)  # Try to convert the value
            except Exception:
                problematic_rows.add(index)

    # Convert the problematic rows to a list and sort them
    problematic_rows = sorted(list(problematic_rows))

    # Drop the problematic rows
    df_llm.drop(problematic_rows, inplace=True)

    # Identify rows where any cell is of type list
    rows_with_lists = df.applymap(lambda x: isinstance(x, list)).any(axis=1)

    # Drop those rows
    df_llm = df_llm[~rows_with_lists]

    df_llm = df_llm.astype(example_df.dtypes)

df_llm

,M1,M2,M3,M4,M5,M6,M7,M8,M9,M10,...,M21,M22,M23,M24,M25,M26,M27,M28,M29,y
0,8.500000,136.200000,40.100000,24.600000,15.500000,11.900000,20.100000,21.200000,5.600000,18.800000,...,56.800000,30.200000,16.500000,63.600000,42.600000,46.100000,17.800000,18.900000,68.90000,0
1,8.500000,155.200000,48.100000,22.600000,18.200000,12.600000,25.200000,18.300000,5.600000,16.500000,...,55.300000,41.900000,21.300000,64.200000,51.800000,51.400000,17.300000,16.800000,61.60000,0
2,8.900000,136.200000,47.600000,23.900000,18.200000,12.600000,24.100000,41.300000,11.500000,12.600000,...,58.200000,32.400000,17.100000,61.200000,37.900000,41.200000,19.700000,20.900000,51.80000,0
3,8.500000,126.100000,44.600000,23.900000,18.200000,13.900000,21.500000,23.700000,8.100000,21.200000,...,59.800000,37.200000,17.300000,64.500000,38.200000,48.600000,15.700000,17.900000,54.10000,0
4,8.900000,145.200000,44.500000,18.200000,15.600000,9.400000,18.300000,16.200000,6.800000,17.300000,...,55.100000,32.100000,15.200000,67.800000,38.900000,30.600000,11.200000,15.900000,57.30000,0
5,8.100000,118.200000,38.600000,28.300000,15.600000,17.200000,18.900000,23.300000,12.600000,20.500000,...,56.100000,37.300000,19.100000,61.100000,48.200000,47.500000,16.200000,19.300000,56.90000,0
6,9.300000,115.200000,37.500000,18.900000,15.600000,10.800000,23.700000,17.300000,12.100000,20.600000,...,50.700000,32.900000,14.500000,61.600000,48.200000,47.900000,16.200000,18.800000,60.50000,0
7,13.400000,158.900000,43.200000,26.900000,16.100000,7.200000,23.700000,13.500000,7.200000,16.800000,...,46.800000,35.400000,21.400000,43.800000,44.900000,35.200000,14.300000,15.600000,51.50000,0
8,6.400000,112.800000,45.600000,24.300000,15.900000,11.200000,18.900000,18.800000,4.100000,16.500000,...,51.600000,34.400000,15.200000,54.900000,32.700000,48.100000,14.800000,18.200000,53.50000,0
9,13.200000,145.900000,57.500000,30.900000,20.600000,13.700000,25.300000,27.600000,7.800000,14.400000,...,54.300000,39.800000,19.600000,37.200000,26.400000,39.500000,12.900000,16.300000,45.20000,0


In [67]:
df_llm.dtypes

M1     float64
M2     float64
M3     float64
M4     float64
M5     float64
M6     float64
M7     float64
M8     float64
M9     float64
M10    float64
M11    float64
M12    float64
M13    float64
M14    float64
M15    float64
M16    float64
M17    float64
M18    float64
M19    float64
M20    float64
M21    float64
M22    float64
M23    float64
M24    float64
M25    float64
M26    float64
M27    float64
M28    float64
M29    float64
y        int32
dtype: object

# STEP 2: Curation

In [68]:
#from src.curation import data_centric_curation
from cllm.curation import data_centric_curation
X_check = df_llm.drop(columns=['y'])
y_check = df_llm['y'].values.astype(int)

curation_metric = 'aleatoric'
curation_ythresh=0.2
curation_xthresh=0 #adaptive

easy_train, ambig_train, unlearnable_train, Curator_xgb = data_centric_curation(X_train_orig, y_train_orig, X_check, y_check, 
                 curation_metric=curation_metric, retrain=False, nest = 100, 
                 curation_ythresh=curation_ythresh, curation_xthresh=curation_xthresh)

curated_train_ids = np.concatenate((easy_train, ambig_train))
curated_train_ids, unlearnable_train

Using adaptive threshold


(array([ 4,  7,  9, 10,  0,  1,  2,  3,  5,  8, 11, 12, 13], dtype=int64),
 array([6], dtype=int64))

In [69]:
# Create a DataFrame with samples that pass curation (curated_train_ids)
df_curated = df_llm.iloc[curated_train_ids]

# Print information about the curated data
print(f"Shape of original generated data: {df_llm.shape}")
print(f"Shape of curated data: {df_curated.shape}")
print(f"Percentage of data retained after curation: {(len(curated_train_ids) / len(df_llm)) * 100:.2f}%")

# Display distribution of easy and ambiguous samples
print(f"\nNumber of easy samples: {len(easy_train)}")
print(f"Number of ambiguous samples: {len(ambig_train)}")
print(f"Number of unlearnable samples: {len(unlearnable_train)}")

# Calculate class distribution in curated data
y_curated = df_curated['y'].values
class_distribution = np.bincount(y_curated) / len(y_curated)
print("\nClass distribution in curated data:")
for i, prob in enumerate(class_distribution):
    print(f"Class {i}: {prob:.2f}")

# Optionally, save the curated data to a CSV file
df_curated.to_csv("rcc_samples.csv", index=False)
print("\nCurated samples saved to 'rcc_samples.csv'")


Shape of original generated data: (14, 30)
Shape of curated data: (13, 30)
Percentage of data retained after curation: 92.86%

Number of easy samples: 4
Number of ambiguous samples: 9
Number of unlearnable samples: 1

Class distribution in curated data:
Class 0: 1.00

Curated samples saved to 'rcc_samples.csv'


In [ ]:
df_curated = df_curated.sample(n=needed_synthetic)

orig = pd.read_csv("mtbls161/mtbls161.csv")
print(orig.shape[0])
appendable_curated = df_curated.rename(columns={'y': 'Class'})
appendable_curated['SampleID'] = "Synthetic Sample"
appendable_curated['SampleType'] = 'serum'
appendable_curated['Chronic Fatigue Syndrome'] = np.where(
    appendable_curated['Class'] == 1, 
    'CFS', 
    'non-CFS control'
)

orig = orig.applymap(lambda x: float(x.replace(',', '')) if isinstance(x, str) and x.replace(',', '').replace('.', '').isdigit() else x)
print(appendable_curated.shape[0])
appendable_curated.to_csv("mtbls161/mtbls161_synthetic.csv")

balanced = pd.concat([orig, appendable_curated], ignore_index=True)
print(balanced.shape[0])
balanced.to_csv("mtbls161/mtbls161_balanced.csv")

59
9
68


In [71]:
count_0 = (balanced['Class'] == 0).sum()
count_1 = (balanced['Class'] == 1).sum()

print(f"Count of y=0: {count_0}")
print(f"Count of y=1: {count_1}")
df_curated

Count of y=0: 34
Count of y=1: 34


,M1,M2,M3,M4,M5,M6,M7,M8,M9,M10,...,M21,M22,M23,M24,M25,M26,M27,M28,M29,y
9,13.2,145.9,57.5,30.9,20.6,13.7,25.3,27.6,7.8,14.4,...,54.3,39.8,19.6,37.2,26.4,39.5,12.9,16.3,45.2,0
7,13.4,158.9,43.2,26.9,16.1,7.2,23.7,13.5,7.2,16.8,...,46.8,35.4,21.4,43.8,44.9,35.2,14.3,15.6,51.5,0
10,8.2,123.6,45.9,24.5,19.2,9.6,23.7,15.2,4.1,16.8,...,65.1,38.4,19.8,64.2,38.9,51.3,18.6,23.6,68.1,0
4,8.9,145.2,44.5,18.2,15.6,9.4,18.3,16.2,6.8,17.3,...,55.1,32.1,15.2,67.8,38.9,30.6,11.2,15.9,57.3,0
5,8.1,118.2,38.6,28.3,15.6,17.2,18.9,23.3,12.6,20.5,...,56.1,37.3,19.1,61.1,48.2,47.5,16.2,19.3,56.9,0
3,8.5,126.1,44.6,23.9,18.2,13.9,21.5,23.7,8.1,21.2,...,59.8,37.2,17.3,64.5,38.2,48.6,15.7,17.9,54.1,0
2,8.9,136.2,47.6,23.9,18.2,12.6,24.1,41.3,11.5,12.6,...,58.2,32.4,17.1,61.2,37.9,41.2,19.7,20.9,51.8,0
13,7.2,112.5,48.1,28.5,19.7,13.9,22.8,24.4,7.1,21.3,...,59.5,34.5,19.2,56.6,40.9,48.1,18.7,25.6,62.4,0
12,6.4,123.5,42.1,25.3,17.8,15.1,23.6,24.9,5.7,19.5,...,55.6,37.8,18.9,61.2,39.2,46.7,17.2,23.1,59.8,0
